# New DSAS workflow (v2 update)

This notebook builds an `NZCCDv2` **subset** from `NZCCDv1`, adds shoreline features for the AOIs picked up by the same new-data selection logic used in the new transect and uncertainty workflows, then runs DSAS-style calculations using the transects and `Total_UNCY` values produced by the two earlier notebooks.

Three things to know about the outputs:

- **They go in your own folder.** Set `RUN_OWNER` to your name and everything lands in `DataUpdatev2/<yourname>/`. Use the same value in all three notebooks. This is what stops two people who run the same AOI from overwriting each other.
- **Only the area you ran is kept.** NZCCDv1 rows outside the matched AOIs are dropped, so the file is your slice of the coast, not the whole country.
- **Filenames are tagged with your selection.** The tag comes from `search_mode`: `region`/`aoi` add the name, the `*_in_date_range` modes also add `since<YYYYMMDD>`.

Outputs (where `<tag>` is e.g. `Auckland_since20240718`):

- `DataUpdatev2/<yourname>/NZCCDv2_<tag>.shp` - shorelines for the AOIs in this run
- `DataUpdatev2/<yourname>/ratesv2_<tag>.shp` - transect-level DSAS rates + date/distance timeseries
- `DataUpdatev2/<yourname>/intersectsv2_<tag>.shp` - transect-shoreline intersection points + attributes
- `DataUpdatev2/<yourname>/new_dsas_exclusions_<tag>.csv` - shorelines left out of DSAS, and why

These per-area files are combined into the national dataset by `NZCCDv2_merge.ipynb`, which is run by the project maintainer.


In [7]:
%load_ext autotime
import warnings
import re
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
import statsmodels.api as sm
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

SOURCE_DIR = Path("Data for testing")

# EDIT TARGET SHORELINES HERE
# See Part 6 of GETTING_STARTED.md; use the same RUN_OWNER and search criteria in all 3 notebooks.
RUN_OWNER = "catriona"
DATA_DIR = Path("DataUpdatev2") / RUN_OWNER
DATA_DIR.mkdir(parents=True, exist_ok=True)
V1_PATH = SOURCE_DIR / "NZCCDv1.shp"
TRANSECTS_PATH = DATA_DIR / "new_transects.shp"
UNCY_SUMMARY_PATH = DATA_DIR / "new_uncy_summary.csv"
# NZCCDv2 / rates / intersects filenames are built from the selection below, in the next cell

# The cutoff can be either a single date (e.g. "2024-07-18") or a date range as a
# two-item tuple/list (e.g. ("2024-07-18", "2024-08-18")).
cutoff_date = ("2024-07-18")
search_roots = [Path(r"Z:\MaxarImagery\HighFreq"), Path(r"Z:\Retrolens")]
search_mode = 'aoi'  # 'date', 'aoi', 'aoi_in_date_range', 'region', or 'region_in_date_range'
target_aoi = ["TeAtatu","Hobsonville","PollenIsland","ShoalBay","SoldiersBay","Ngataringa","FrenchmansBay","ManukapuaIsland","OmokoritoBay","OrongoPoint","Pouto","ShellyBeach","TeHakono_clarksBay","Tinopai"]
target_region = "Auckland"


def _norm(text):
    return ''.join(ch for ch in str(text).lower() if ch.isalnum())


def _coerce_cutoff_bounds(cutoff):
    if cutoff is None:
        raise ValueError("cutoff_date must not be None")

    if isinstance(cutoff, (tuple, list)):
        if len(cutoff) != 2:
            raise ValueError("cutoff_date range must be a two-item tuple/list")
        start = pd.Timestamp(cutoff[0]).normalize()
        end = pd.Timestamp(cutoff[1]).normalize()
        if start > end:
            start, end = end, start
        return start, end

    single = pd.Timestamp(cutoff).normalize()
    return single, None


def _matches_date(modified, cutoff_start, cutoff_end):
    if cutoff_end is None:
        return modified > cutoff_start
    return (modified >= cutoff_start) and (modified <= cutoff_end)


def _cutoff_label(cutoff_start, cutoff_end):
    if cutoff_end is None:
        return f"modified > {cutoff_start.date()}"
    return f"modified between {cutoff_start.date()} and {cutoff_end.date()}"


def _date_tag(cutoff):
    start, end = _coerce_cutoff_bounds(cutoff)
    if end is None:
        return f"since{start:%Y%m%d}"
    return f"since{start:%Y%m%d}to{end:%Y%m%d}"


def normalize_path(value):
    return str(value).replace('\\', '/').lower()


def pick_col(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None


def parse_date_from_stem(stem):
    text = str(stem)
    m = re.search(r"(\d{1,2}[A-Za-z]{3,4}\d{4})", text)
    if m:
        token = m.group(1).upper().replace("APRL", "APR").replace("SEPT", "SEP")
        for fmt in ("%d%b%Y", "%d%B%Y"):
            try:
                return pd.to_datetime(token, format=fmt)
            except Exception:
                pass
    m = re.search(r"(\d{4}-\d{2}-\d{2})", text)
    if m:
        try:
            return pd.to_datetime(m.group(1), format="%Y-%m-%d")
        except Exception:
            pass
    m = re.search(r"(\d{8})", text)
    if m:
        try:
            return pd.to_datetime(m.group(1), format="%Y%m%d")
        except Exception:
            pass
    return pd.NaT


def geom_hash(geom):
    if geom is None:
        return None
    try:
        return shapely.to_wkb(geom, hex=True)
    except Exception:
        return None

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 0 ns (started: 2026-08-19 17:12:10 +12:00)


In [8]:
# 1) Name the outputs after the selection, then load NZCCDv1 as the starting point
# Tagging filenames keeps two people working on different areas from writing to the same file.
def _slug(text):
    return re.sub(r"[^A-Za-z0-9]+", "", str(text))

cutoff_start, cutoff_end = _coerce_cutoff_bounds(cutoff_date)
_date_tag = _date_tag(cutoff_date)
if search_mode == "date":
    OUTPUT_TAG = _date_tag
elif search_mode == "aoi":
    OUTPUT_TAG = _slug(target_aoi)
elif search_mode == "aoi_in_date_range":
    OUTPUT_TAG = f"{_slug(target_aoi)}_{_date_tag}"
elif search_mode == "region":
    OUTPUT_TAG = _slug(target_region)
elif search_mode == "region_in_date_range":
    OUTPUT_TAG = f"{_slug(target_region)}_{_date_tag}"
else:
    raise ValueError(f"Unknown search_mode: {search_mode}")

V2_PATH = DATA_DIR / f"NZCCDv2_{OUTPUT_TAG}.shp"
RATES_OUT = DATA_DIR / f"ratesv2_{OUTPUT_TAG}.shp"
POINTS_OUT = DATA_DIR / f"intersectsv2_{OUTPUT_TAG}.shp"
EXCLUSIONS_OUT = DATA_DIR / f"new_dsas_exclusions_{OUTPUT_TAG}.csv"

if not V1_PATH.exists():
    raise FileNotFoundError(f"Missing source dataset: {V1_PATH}")

v2 = gpd.read_file(V1_PATH)

print(f"Output tag: {OUTPUT_TAG}")
print(f"Will write: {V2_PATH.name} / {RATES_OUT.name} / {POINTS_OUT.name}")
print(f"Loaded NZCCDv1 rows: {len(v2):,}")
v2.head(2)


Output tag: TeAtatuHobsonvillePollenIslandShoalBaySoldiersBayNgataringaFrenchmansBayManukapuaIslandOmokoritoBayOrongoPointPoutoShellyBeachTeHakonoclarksBayTinopai
Will write: NZCCDv2_TeAtatuHobsonvillePollenIslandShoalBaySoldiersBayNgataringaFrenchmansBayManukapuaIslandOmokoritoBayOrongoPointPoutoShellyBeachTeHakonoclarksBayTinopai.shp / ratesv2_TeAtatuHobsonvillePollenIslandShoalBaySoldiersBayNgataringaFrenchmansBayManukapuaIslandOmokoritoBayOrongoPointPoutoShellyBeachTeHakonoclarksBayTinopai.shp / intersectsv2_TeAtatuHobsonvillePollenIslandShoalBaySoldiersBayNgataringaFrenchmansBayManukapuaIslandOmokoritoBayOrongoPointPoutoShellyBeachTeHakonoclarksBayTinopai.shp
Loaded NZCCDv1 rows: 19,666


,Region,Site,Digitiser,Scale,Notes,Source,CPS,Proxy,Photoscale,Georef_ER,Pixel_Er,Total_UNCY,USDate,SHLength,Date,ID,geometry
0,Auckland,KarekareBethells,MW,2000,tod,RL,4,1,40000,5.03,1.416425,5.620681,01/02/2004,1.265403,2004-01-02,0.0,"LINESTRING Z (1728907.5 5916213.248 0, 1728868.341 5916181.498 0, 1728823.891 5916158.215 0, 1728765.154 5916132.815 0, 1728735.52 59161..."
1,Auckland,KarekareBethells,MW,2000,tod,RL,4,1,40000,5.03,1.416425,5.620681,01/02/2004,0.307010,2004-01-02,1.0,"LINESTRING Z (1729067.838 5914779.733 0, 1729077.892 5914779.733 0, 1729089.534 5914776.028 0, 1729092.179 5914765.445 0, 1729082.654 59..."


time: 390 ms (started: 2026-08-19 17:12:14 +12:00)


In [6]:
# 2) Find new shoreline files using the same mode logic as new_transects/new_uncy
valid_modes = {'date', 'aoi', 'aoi_in_date_range', 'region', 'region_in_date_range'}
if search_mode not in valid_modes:
    raise ValueError(f"search_mode must be one of {sorted(valid_modes)}")
if search_mode in {'aoi', 'aoi_in_date_range'} and not str(target_aoi).strip():
    raise ValueError('target_aoi must be set when using AOI-based modes')
if search_mode in {'region', 'region_in_date_range'} and not str(target_region).strip():
    raise ValueError('target_region must be set when using region-based modes')

cutoff_start, cutoff_end = _coerce_cutoff_bounds(cutoff_date)
target_aoi_norm = _norm(target_aoi)
target_region_norm = _norm(target_region)
records = []

for root in search_roots:
    if not root.exists():
        continue
    for shp in root.glob('**/Shorelines/*.shp'):
        if shp.stem.lower().startswith('[aoierr]'):
            continue
        if len(shp.parts) < 5 or shp.parts[-2].lower() != 'shorelines':
            continue

        region = shp.parts[-4]
        aoi = shp.parts[-3]
        modified = pd.Timestamp(shp.stat().st_mtime, unit='s')
        stem_aoi = shp.stem.rsplit('_', 1)[0]

        matches_aoi = target_aoi_norm in {_norm(aoi), _norm(stem_aoi)}
        matches_region = target_region_norm == _norm(region)
        matches_date = _matches_date(modified, cutoff_start, cutoff_end)

        include = False
        if search_mode == 'date':
            include = matches_date
        elif search_mode == 'aoi':
            include = matches_aoi
        elif search_mode == 'aoi_in_date_range':
            include = matches_aoi and matches_date
        elif search_mode == 'region':
            include = matches_region
        elif search_mode == 'region_in_date_range':
            include = matches_region and matches_date

        if include:
            records.append({
                'region': region,
                'aoi': aoi,
                'shoreline_path': str(shp),
                'modified': modified,
            })

new_shorelines = pd.DataFrame(records)
if new_shorelines.empty:
    raise ValueError('No shoreline files matched the selected search criteria')

new_shorelines = new_shorelines.sort_values(['region', 'aoi', 'modified']).reset_index(drop=True)
target_aois = new_shorelines[['region', 'aoi']].drop_duplicates().reset_index(drop=True)

print(f"Matched new shoreline files: {len(new_shorelines)}")
print(f"Target AOIs for trimming NZCCDv1: {len(target_aois)}")
new_shorelines.head(20)

ValueError: No shoreline files matched the selected search criteria

time: 2min 25s (started: 2026-08-19 17:06:55 +12:00)


In [9]:
# Normalize target selection to the same list-aware matching used by new_transects/new_uncy.
def _as_target_list(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(item).strip() for item in value if str(item).strip()]
    return [str(value).strip()] if str(value).strip() else []

search_aoi_norms = {_norm(item) for item in _as_target_list(target_aoi)}
search_region_norms = {_norm(item) for item in _as_target_list(target_region)}
records = []

for root in search_roots:
    if not root.exists():
        continue
    for shp in root.glob('**/Shorelines/*.shp'):
        if shp.stem.lower().startswith('[aoierr]') or len(shp.parts) < 5:
            continue
        region, aoi = shp.parts[-4], shp.parts[-3]
        modified = pd.Timestamp(shp.stat().st_mtime, unit='s')
        stem_aoi = shp.stem.rsplit('_', 1)[0]
        matches_aoi = bool(search_aoi_norms & {_norm(aoi), _norm(stem_aoi)})
        matches_region = _norm(region) in search_region_norms
        matches_date = _matches_date(modified, cutoff_start, cutoff_end)
        include = (
            search_mode == 'date' and matches_date
            or search_mode == 'aoi' and matches_aoi
            or search_mode == 'aoi_in_date_range' and matches_aoi and matches_date
            or search_mode == 'region' and matches_region
            or search_mode == 'region_in_date_range' and matches_region and matches_date
        )
        if include:
            records.append({'region': region, 'aoi': aoi, 'shoreline_path': str(shp), 'modified': modified})

new_shorelines = pd.DataFrame(records)
if new_shorelines.empty:
    raise ValueError('No shoreline files matched the selected search criteria')
new_shorelines = new_shorelines.sort_values(['region', 'aoi', 'modified']).reset_index(drop=True)
target_aois = new_shorelines[['region', 'aoi']].drop_duplicates().reset_index(drop=True)
print(f'Final target search: {len(new_shorelines)} shoreline files across {len(target_aois)} AOI(s)')


Final target search: 67 shoreline files across 14 AOI(s)
time: 2min 26s (started: 2026-08-19 17:12:16 +12:00)


In [10]:
# 2b) Trim NZCCDv1 down to the areas this run actually covers
# Rows outside the matched AOIs are dropped, so the NZCCDv2 written below holds only
# the coastline this run is responsible for and can be merged with other people's runs later.
region_col = pick_col(v2, ["Region"])
location_col = pick_col(v2, ["Location", "Site"])
if region_col is None or location_col is None:
    raise ValueError("NZCCDv1 has no Region/Location(Site) columns, so it cannot be trimmed to the target AOIs")

target_pairs = {(_norm(r.region), _norm(r.aoi)) for r in target_aois.itertuples(index=False)}
v2_pairs = list(zip(v2[region_col].map(_norm), v2[location_col].map(_norm)))
keep_mask = pd.Series([pair in target_pairs for pair in v2_pairs], index=v2.index)

kept_pairs = {pair for pair, keep in zip(v2_pairs, keep_mask) if keep}
missing_pairs = sorted(target_pairs - kept_pairs)

dropped = int((~keep_mask).sum())
v2 = v2[keep_mask].reset_index(drop=True)

# NZCCDv1 calls the AOI column "Site"; the merge/dedupe steps below expect "Location"
if "Location" not in v2.columns:
    v2["Location"] = v2[location_col]

print(f"Kept {len(v2):,} NZCCDv1 rows across {len(kept_pairs)} AOI(s); dropped {dropped:,} rows outside this run")
if missing_pairs:
    print(f"No NZCCDv1 history for {len(missing_pairs)} target AOI(s) - those shorelines come from the drive only:")
    for region, aoi in missing_pairs:
        print(f"  - {region} / {aoi}")
if v2.empty:
    print("WARNING: no NZCCDv1 rows matched. Check that Region/Site spelling in NZCCDv1 matches the drive folder names.")


Kept 0 NZCCDv1 rows across 0 AOI(s); dropped 19,666 rows outside this run
No NZCCDv1 history for 14 target AOI(s) - those shorelines come from the drive only:
  - auckland / hobsonville
  - auckland / manukapuaisland
  - auckland / ngataringa
  - auckland / omokoritobay
  - auckland / orongopoint
  - auckland / pollenisland
  - auckland / shellybeach
  - auckland / shoalbay
  - auckland / soldiersbay
  - auckland / teatatu
  - auckland / tehakonoclarksbay
  - northland / frenchmansbay
  - northland / pouto
  - northland / tinopai
time: 47 ms (started: 2026-08-19 17:14:44 +12:00)


In [11]:
# 3) Build the DSAS shoreline table from the target search only.
# Read one selected file at a time and keep only fields needed by DSAS.
UNCY_ROW_REPORT_PATH = DATA_DIR / 'new_uncy_row_report.csv'
row_report = pd.DataFrame()
if UNCY_ROW_REPORT_PATH.exists():
    row_report = pd.read_csv(UNCY_ROW_REPORT_PATH)
    row_report['path_norm'] = row_report['shoreline_path'].map(normalize_path)
    row_report['row_number'] = pd.to_numeric(row_report['row_number'], errors='coerce')
    row_report['total_uncy'] = pd.to_numeric(row_report['total_uncy'], errors='coerce')
    row_report['dsas_date'] = pd.to_datetime(row_report['dsas_date'], errors='coerce')
else:
    print(f'WARNING: no row-level uncertainty report at {UNCY_ROW_REPORT_PATH}')

needed_cols = ['DSAS_Date', 'Date', 'Total_UNCY', 'SourceFile', 'GeomHash', 'geometry']
source_frames = []
exclusion_rows = []

for rec in tqdm(new_shorelines.to_dict('records'), total=len(new_shorelines), desc='Load selected shorelines'):
    shp = Path(rec['shoreline_path'])
    try:
        g = gpd.read_file(shp)
    except Exception as error:
        exclusion_rows.append({'filename': str(shp), 'region': rec['region'], 'aoi': rec['aoi'], 'reason': f'Could not read: {error}'})
        continue
    if g.empty:
        exclusion_rows.append({'filename': str(shp), 'region': rec['region'], 'aoi': rec['aoi'], 'reason': 'Empty shapefile'})
        continue

    date_col = pick_col(g, ['DSAS_Date', 'Date'])
    if date_col is None:
        g['Date'] = parse_date_from_stem(shp.stem)
    else:
        g['Date'] = pd.to_datetime(g[date_col], errors='coerce')

    source_norm = normalize_path(shp)
    report = row_report[row_report['path_norm'] == source_norm].set_index('row_number') if not row_report.empty else pd.DataFrame()
    if not report.empty:
        report_dates = report['dsas_date'].to_dict()
        report_uncy = report['total_uncy'].to_dict()
        g['Date'] = [report_dates.get(index, value) for index, value in zip(g.index, g['Date'])]
        g['Total_UNCY'] = [report_uncy.get(index, np.nan) for index in g.index]
    else:
        uncy_col = pick_col(g, ['Total_UNCY', 'total_uncy'])
        g['Total_UNCY'] = pd.to_numeric(g[uncy_col], errors='coerce') if uncy_col else np.nan

    g['Region'] = rec['region']
    g['Location'] = rec['aoi']
    g['SourceFile'] = str(shp)
    g['GeomHash'] = g.geometry.apply(geom_hash)
    keep = [column for column in ['Region', 'Location', 'Date', 'Total_UNCY', 'SourceFile', 'GeomHash', 'geometry'] if column in g.columns]
    source_frames.append(g[keep].copy())

if not source_frames:
    raise ValueError('No selected shoreline features could be loaded.')

incoming = gpd.GeoDataFrame(pd.concat(source_frames, ignore_index=True), crs=source_frames[0].crs)
v2_updated = incoming
v2_updated['Date'] = pd.to_datetime(v2_updated['Date'], errors='coerce')
v2_updated['Total_UNCY'] = pd.to_numeric(v2_updated['Total_UNCY'], errors='coerce')
v2_updated.to_file(V2_PATH)

dsas_exclusions = pd.DataFrame(exclusion_rows)
print(f'Loaded {len(v2_updated):,} selected shoreline rows from {len(source_frames):,} files')
print(f'Dated rows: {v2_updated["Date"].notna().sum():,}; rows with uncertainty: {v2_updated["Total_UNCY"].notna().sum():,}')


Load selected shorelines:   0%|          | 0/67 [00:00<?, ?it/s]

Loaded 1,121 selected shoreline rows from 66 files
Dated rows: 1,121; rows with uncertainty: 1,121
time: 8.91 s (started: 2026-08-19 17:14:46 +12:00)


c:\envs\coastal\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field Date created as String field, though DateTime requested.
  ogr_write(
c:\envs\coastal\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value '01020000002A00000030772DC1D7B03A418638D6A1EA9B5641085F98FCD8B03A41C4B12E8AEA9B5641F4285CFFDAB03A4135EF384DEA9B564184EB5138DDB03A412041F1FBE99B5641D8A3707DDEB03A41F7E46159E99B56411895D4E9DEB03A41B1E1E9D1E89B564114AE4771DFB03A41B003E73CE89B56416C4DF32EE0B03A41EEEBC0C9E79B5641B81E85FBE1B03A416ADE714EE69B5641ACCFD576E3B03A41D95F76E7E49B56418C4AEAF4E6B03A4136CD3BF2E29B5641085F98DCE8B03A414B5986DCE19B564160764FEEEBB03A41925CFE67E09B5641B02E6E33EDB03A41F085C99CDF9B56418C4AEAE4EEB03A41BD5296D1DE9B564134A2B4E7F0B03A414182E2BBDD9B564188D2DE80F4B03A416ADE7162DC9B5641DC8AFDC5F5B03A41DDB584E8DB9B5641E002091AF8B03A41FA7E6A38DB9B5641246C78DAFAB03A416C787A29DA9B5641FC87F48BFCB03A410E2DB28DD99B564158863816FFB03A41386744C9D89B5641AC3E575B00B13A4194F6063BD89B5641B415FB7B04B13A41423EE8F

In [ ]:
# Normalize dates from the source shoreline attributes before DSAS reloads NZCCDv2.
# SourceFile plus geometry hash lets merged files retain each row's own DSAS date.
if 'SourceFile' in v2_updated.columns:
    for source_file in v2_updated['SourceFile'].dropna().astype(str).unique():
        source_path = Path(source_file)
        if not source_path.exists():
            continue
        source_rows = gpd.read_file(source_path)
        source_date_col = pick_col(source_rows, ['DSAS_Date', 'Date'])
        if source_date_col is None:
            source_rows['Date'] = parse_date_from_stem(source_path.stem)
        else:
            source_rows['Date'] = pd.to_datetime(source_rows[source_date_col], errors='coerce')
        source_rows['GeomHash'] = source_rows.geometry.apply(geom_hash)
        date_by_geom = source_rows.dropna(subset=['GeomHash']).drop_duplicates('GeomHash').set_index('GeomHash')['Date'].to_dict()

        source_mask = v2_updated['SourceFile'].astype(str).eq(source_file)
        row_dates = v2_updated.loc[source_mask, 'GeomHash'].map(date_by_geom)
        v2_updated.loc[source_mask, 'Date'] = row_dates.values

v2_updated['Date'] = pd.to_datetime(v2_updated['Date'], errors='coerce')
v2_updated.to_file(V2_PATH)
print(f'Normalized NZCCDv2 dates from DSAS_Date/Date attributes: {v2_updated["Date"].notna().sum():,} dated rows')


In [ ]:
# Apply the row-level uncertainty report to the DSAS working table.
# This uses DSAS_Date and Total_UNCY from new_uncy without writing source shorelines.
UNCY_ROW_REPORT_PATH = DATA_DIR / 'new_uncy_row_report.csv'
if not UNCY_ROW_REPORT_PATH.exists():
    print(f'WARNING: no row-level uncertainty report at {UNCY_ROW_REPORT_PATH}')
else:
    row_report = pd.read_csv(UNCY_ROW_REPORT_PATH)
    row_report['path_norm'] = row_report['shoreline_path'].map(normalize_path)
    row_report['row_number'] = pd.to_numeric(row_report['row_number'], errors='coerce')
    row_report['total_uncy'] = pd.to_numeric(row_report['total_uncy'], errors='coerce')
    row_report['dsas_date'] = pd.to_datetime(row_report['dsas_date'], errors='coerce')

    for source_file in v2_updated['SourceFile'].dropna().astype(str).unique():
        source_path = Path(source_file)
        if not source_path.exists():
            continue
        source_rows = gpd.read_file(source_path)
        source_path_norm = normalize_path(source_path)
        source_report = row_report[row_report['path_norm'] == source_path_norm].set_index('row_number')
        if source_report.empty:
            continue
        geom_to_report = {}
        for source_index, source_row in source_rows.iterrows():
            if source_index in source_report.index:
                geom_to_report[geom_hash(source_row.geometry)] = source_report.loc[source_index]

        source_mask = v2_updated['SourceFile'].astype(str).eq(source_file)
        for row_index in v2_updated.index[source_mask]:
            report_row = geom_to_report.get(v2_updated.at[row_index, 'GeomHash'])
            if report_row is None:
                continue
            if pd.notna(report_row['dsas_date']):
                v2_updated.at[row_index, 'Date'] = report_row['dsas_date']
            if pd.notna(report_row['total_uncy']):
                v2_updated.at[row_index, 'Total_UNCY'] = report_row['total_uncy']

    v2_updated['Date'] = pd.to_datetime(v2_updated['Date'], errors='coerce')
    v2_updated['Total_UNCY'] = pd.to_numeric(v2_updated['Total_UNCY'], errors='coerce')
    v2_updated.to_file(V2_PATH)
    print(f'Applied row-level uncertainty/date values from {UNCY_ROW_REPORT_PATH}')


In [12]:
# 4) DSAS calculations using new transects and updated NZCCDv2 shorelines
if not TRANSECTS_PATH.exists():
    raise FileNotFoundError(f"Missing transects: {TRANSECTS_PATH}")

transects = gpd.read_file(TRANSECTS_PATH)
uid_col = pick_col(transects, ['Unique_ID', 'UniqueID'])
if uid_col is None:
    raise ValueError('Transects must include Unique_ID or UniqueID')
transects = transects.rename(columns={uid_col: 'Unique_ID'}).set_index('Unique_ID')
if transects.crs is None:
    transects = transects.set_crs(2193, allow_override=True)
else:
    transects = transects.to_crs(2193)

# Ensure one transect geometry per Unique_ID.
if transects.index.duplicated().any():
    dup_count = int(transects.index.duplicated().sum())
    print(f"Dropping {dup_count:,} duplicate transect rows by Unique_ID (keeping first geometry)")
    transects = transects[~transects.index.duplicated(keep='first')].copy()

shore = gpd.read_file(V2_PATH)
if shore.crs is None:
    shore = shore.set_crs(2193, allow_override=True)
else:
    shore = shore.to_crs(2193)

region_col = pick_col(shore, ['Region', 'region'])
aoi_col = pick_col(shore, ['Location', 'AOI', 'aoi', 'location'])
date_col = pick_col(shore, ['Date', 'date'])
uncy_col = pick_col(shore, ['Total_UNCY', 'total_uncy', 'new_Total_UNCY'])
sourcefile_col = pick_col(shore, ['SourceFile', 'sourcefile'])

if region_col is None or aoi_col is None or date_col is None:
    raise ValueError('NZCCDv2 must include Region/Location(Date) columns for DSAS')

shore = shore.rename(columns={region_col: 'Region', aoi_col: 'AOI', date_col: 'Date'})
shore = shore.loc[:, ~shore.columns.duplicated()].copy()
shore = shore.set_geometry('geometry')
shore['Date'] = pd.to_datetime(shore['Date'], errors='coerce')
if uncy_col is None:
    shore['Total_UNCY'] = np.nan
else:
    shore['Total_UNCY'] = pd.to_numeric(shore[uncy_col], errors='coerce')
if sourcefile_col is None:
    shore['SourceFile'] = pd.NA
else:
    shore['SourceFile'] = shore[sourcefile_col].astype(str)

target_key = set(target_aois.apply(lambda r: (_norm(r.region), _norm(r.aoi)), axis=1).tolist())
shore = shore[shore.apply(lambda r: (_norm(r['Region']), _norm(r['AOI'])) in target_key, axis=1)].copy()
shore = shore[shore.geometry.notna()].copy()
shore['has_uncy'] = shore['Total_UNCY'].notna() & (shore['Total_UNCY'] > 0)
shore['dsas_eligible'] = shore['Date'].notna()

def to_point_or_empty(geom, transect_origin):
    if geom is None or geom.is_empty:
        return shapely.Point()
    gt = geom.geom_type
    if gt == 'Point':
        return geom
    if gt == 'MultiPoint':
        pts = list(geom.geoms)
        if len(pts) == 0:
            return shapely.Point()
        pts = sorted(pts, key=lambda p: p.distance(transect_origin))
        return pts[0]
    try:
        p = shapely.get_point(geom, 0)
        return p if p is not None else shapely.Point()
    except Exception:
        return shapely.Point()

def intersect_or_empty(geom, line):
    if geom is None:
        return shapely.GeometryCollection()
    try:
        return geom.intersection(line)
    except Exception:
        return shapely.GeometryCollection()

def process_transect(unique_id):
    transect = transects.geometry.loc[unique_id]
    tran_origin = shapely.get_point(transect, -1)

    local = shore.copy()
    intersections = [intersect_or_empty(geom, transect) for geom in local.geometry.values]
    local['intersect_raw'] = intersections
    local['intersect_point'] = [to_point_or_empty(g, tran_origin) for g in intersections]

    # Promote intersection points to a GeoSeries so geometric vector ops are available.
    point_gs = gpd.GeoSeries(local['intersect_point'], index=local.index, crs=shore.crs)
    local = local[~point_gs.is_empty].copy()
    point_gs = point_gs.loc[local.index]
    local = local[local['dsas_eligible']].sort_values('Date')
    point_gs = point_gs.loc[local.index]

    if len(local) < 3:
        return None, None

    local['YearsSinceBase'] = (local['Date'] - local['Date'].min()).dt.days / 365.25
    local['Distance'] = point_gs.distance(tran_origin)

    lr = sm.OLS(local['Distance'], sm.add_constant(local['YearsSinceBase'])).fit()
    lr_low, lr_high = lr.conf_int(alpha=0.1).loc['YearsSinceBase']
    lci = (lr_high - lr_low) / 2

    # Weighted stats need an uncertainty on every shoreline used by this transect.
    weighted = bool(local['has_uncy'].all())
    if weighted:
        wlr = sm.WLS(local['Distance'], sm.add_constant(local['YearsSinceBase']), weights=1 / (local['Total_UNCY'] ** 2)).fit()
        wlr_low, wlr_high = wlr.conf_int(alpha=0.1).loc['YearsSinceBase']
        wci = (wlr_high - wlr_low) / 2
        wlr_stats = {
            'WLR': round(wlr.params['YearsSinceBase'], 2),
            'WLI': round(wlr.params['const'], 2),
            'WCI': round(wci, 2),
            'WSE': round(np.sqrt(wlr.mse_resid), 2),
            'WR2': round(wlr.rsquared, 2),
        }
    else:
        wlr_stats = {'WLR': np.nan, 'WLI': np.nan, 'WCI': np.nan, 'WSE': np.nan, 'WR2': np.nan}

    duration = (local['Date'].max() - local['Date'].min()).days / 365.25
    if duration <= 0:
        return None, None

    nsm = -(local['Distance'].iloc[0] - local['Distance'].iloc[-1])
    sce = point_gs.apply(lambda p: point_gs.distance(p).max()).max()

    first_uncy = local['Total_UNCY'].iloc[0]
    last_uncy = local['Total_UNCY'].iloc[-1]
    epr_unc = (
        round(np.sqrt(first_uncy ** 2 + last_uncy ** 2) / duration, 2)
        if pd.notna(first_uncy) and pd.notna(last_uncy)
        else np.nan
    )

    rate_row = {
        'UniqueID': unique_id,
        'Region': local['Region'].astype(str).value_counts().idxmax(),
        'AOI': local['AOI'].astype(str).value_counts().idxmax(),
        'Start_date': str(local['Date'].min().date()),
        'End_date': str(local['Date'].max().date()),
        'Duration': round(duration),
        'ShrCount': len(local),
        'UncyCount': int(local['has_uncy'].sum()),
        'NSM': round(nsm, 2),
        'SCE': round(sce, 2),
        'EPR': round(nsm / duration, 2),
        'EPRunc': epr_unc,
        'LRR': round(lr.params['YearsSinceBase'], 2),
        'LRI': round(lr.params['const'], 2),
        'LCI': round(lci, 2),
        'LSE': round(np.sqrt(lr.mse_resid), 2),
        'LR2': round(lr.rsquared, 2),
        **wlr_stats,
        'Dates': local['Date'].dt.strftime('%Y-%m-%d').tolist(),
        'Distances': local['Distance'].round(2).tolist(),
        'geometry': transect,
    }

    point_rows = local[['Region', 'AOI', 'Date', 'Distance', 'Total_UNCY', 'SourceFile', 'intersect_point']].copy()
    point_rows['Unique_ID'] = unique_id
    point_rows['YearsSinceBase'] = local['YearsSinceBase']
    point_rows['NSM'] = round(nsm, 2)
    point_rows['EPR'] = round(nsm / duration, 2)
    point_rows['LRR'] = round(lr.params['YearsSinceBase'], 2)
    point_rows['WLR'] = wlr_stats['WLR']
    point_rows = point_rows.rename(columns={'intersect_point': 'geometry'})

    return rate_row, point_rows

rate_rows = []
point_frames = []
for uid in tqdm(transects.index.tolist()):
    rate_row, point_rows = process_transect(uid)
    if rate_row is None:
        continue
    rate_rows.append(rate_row)
    point_frames.append(point_rows)

if len(rate_rows) == 0:
    raise ValueError('No transects produced DSAS statistics (need at least 3 eligible shoreline intersections per transect).')

rates = gpd.GeoDataFrame(rate_rows, crs=transects.crs)
points = gpd.GeoDataFrame(pd.concat(point_frames, ignore_index=True), crs=transects.crs)
points['Date'] = pd.to_datetime(points['Date'], errors='coerce')

# Add post-DSAS exclusions for eligible shoreline files that never made it into DSAS outputs.
if 'dsas_exclusions' not in globals():
    dsas_exclusions = pd.DataFrame(columns=['filename', 'region', 'aoi', 'is_new_shoreline', 'total_rows', 'excluded_rows', 'included_rows', 'rows_without_uncy', 'reason'])

eligible_files = set(
    shore.loc[shore['dsas_eligible'] & shore['SourceFile'].notna() & (shore['SourceFile'].astype(str) != 'nan'), 'SourceFile']
    .astype(str)
    .unique()
    .tolist()
)
used_files = set(points['SourceFile'].dropna().astype(str).unique().tolist())
not_used_files = sorted(eligible_files - used_files)

if len(not_used_files) > 0:
    extra_exclusions = pd.DataFrame({
        'filename': not_used_files,
        'region': [pd.NA] * len(not_used_files),
        'aoi': [pd.NA] * len(not_used_files),
        'is_new_shoreline': [pd.NA] * len(not_used_files),
        'total_rows': [pd.NA] * len(not_used_files),
        'excluded_rows': [pd.NA] * len(not_used_files),
        'included_rows': [pd.NA] * len(not_used_files),
        'rows_without_uncy': [pd.NA] * len(not_used_files),
        'reason': ['eligible shoreline not used in final DSAS results (no successful transect intersections)'] * len(not_used_files),
    })
    dsas_exclusions = pd.concat([dsas_exclusions, extra_exclusions], ignore_index=True).drop_duplicates()

display(rates.head(10))
display(points.head(10))
print(f"Rates rows: {len(rates):,}")
print(f"Point rows: {len(points):,}")
print(f"DSAS exclusions rows: {len(dsas_exclusions):,}")

  0%|          | 0/4298 [00:00<?, ?it/s]

,UniqueID,Region,AOI,Start_date,End_date,Duration,ShrCount,UncyCount,NSM,SCE,EPR,EPRunc,LRR,LRI,LCI,LSE,LR2,WLR,WLI,WCI,WSE,WR2,Dates,Distances,geometry
0,100227997741,Auckland,Hobsonville,1940-04-22,2024-01-01,84,4,4,-8.68,13.15,-0.10,0.03,-0.14,145.22,0.10,2.46,0.90,-0.12,143.25,0.24,4.09,0.50,"[1940-04-22, 1950-10-14, 2017-01-01, 2024-01-01]","[143.75, 145.63, 132.47, 135.07]","LINESTRING (1749436.237 5926953.508, 1749113.267 5926717.521)"
1,100227998951,Auckland,Hobsonville,1940-04-22,2024-01-01,84,4,4,-9.52,11.79,-0.11,0.03,-0.14,143.56,0.07,1.68,0.95,-0.12,142.09,0.18,3.02,0.66,"[1940-04-22, 1950-10-14, 2017-01-01, 2024-01-01]","[142.68, 143.26, 131.47, 133.15]","LINESTRING (1749442.136 5926945.434, 1749119.166 5926709.446)"
2,100227999629,Auckland,Hobsonville,1940-04-22,2024-01-01,84,4,4,-9.36,10.93,-0.11,0.03,-0.13,141.34,0.05,1.41,0.96,-0.12,140.04,0.16,2.65,0.70,"[1940-04-22, 1950-10-14, 2017-01-01, 2024-01-01]","[140.68, 140.83, 129.91, 131.32]","LINESTRING (1749448.035 5926937.359, 1749125.065 5926701.371)"
3,100228000306,Auckland,Hobsonville,1940-04-22,2024-01-01,84,4,4,-7.23,8.25,-0.09,0.03,-0.10,137.82,0.04,0.96,0.97,-0.09,136.86,0.11,1.93,0.72,"[1940-04-22, 1950-10-14, 2017-01-01, 2024-01-01]","[137.46, 137.26, 129.21, 130.23]","LINESTRING (1749453.934 5926929.284, 1749130.964 5926693.296)"
4,100228000764,Auckland,Hobsonville,1940-04-22,2024-01-01,84,4,4,-6.62,10.27,-0.08,0.03,-0.11,138.33,0.08,2.02,0.89,-0.10,137.18,0.15,2.51,0.64,"[1940-04-22, 1950-10-14, 2017-01-01, 2024-01-01]","[136.79, 139.08, 128.81, 130.17]","LINESTRING (1749459.833 5926921.209, 1749136.863 5926685.221)"
5,100228001307,Auckland,Hobsonville,1940-04-22,2024-01-01,84,4,4,-5.59,6.31,-0.07,0.03,-0.07,135.19,0.03,0.78,0.96,-0.06,134.50,0.08,1.36,0.70,"[1940-04-22, 1950-10-14, 2017-01-01, 2024-01-01]","[135.68, 133.97, 129.37, 130.09]","LINESTRING (1749465.732 5926913.134, 1749142.762 5926677.147)"
6,100228001836,Auckland,Hobsonville,1940-04-22,2024-01-01,84,4,4,-5.35,6.48,-0.06,0.03,-0.06,135.02,0.05,1.18,0.89,-0.05,134.12,0.11,1.80,0.50,"[1940-04-22, 1950-10-14, 2017-01-01, 2024-01-01]","[135.88, 133.45, 129.4, 130.53]","LINESTRING (1749471.631 5926905.06, 1749148.661 5926669.072)"
7,100228002363,Auckland,Hobsonville,1940-04-22,2024-01-01,84,4,4,-8.17,9.29,-0.10,0.03,-0.07,137.10,0.12,3.22,0.61,-0.06,136.10,0.14,2.31,0.45,"[1940-04-22, 1950-10-14, 2017-01-01, 2024-01-01]","[140.02, 133.03, 130.73, 131.85]","LINESTRING (1749477.53 5926896.985, 1749154.56 5926660.997)"
8,100228003092,Auckland,Hobsonville,1940-04-22,2024-01-01,84,4,4,-8.64,9.99,-0.10,0.03,-0.08,138.97,0.14,3.57,0.58,-0.06,137.82,0.16,2.63,0.40,"[1940-04-22, 1950-10-14, 2017-01-01, 2024-01-01]","[142.2, 134.53, 132.21, 133.56]","LINESTRING (1749483.429 5926888.91, 1749160.459 5926652.922)"
9,100228003821,Auckland,Hobsonville,1940-04-22,2024-01-01,84,4,4,-5.95,5.97,-0.07,0.03,-0.05,139.07,0.07,1.92,0.69,-0.05,138.81,0.06,0.98,0.75,"[1940-04-22, 1950-10-14, 2017-01-01, 2024-01-01]","[140.85, 136.49, 134.88, 134.9]","LINESTRING (1749489.328 5926880.835, 1749166.358 5926644.847)"


,Region,AOI,Date,Distance,Total_UNCY,SourceFile,geometry,Unique_ID,YearsSinceBase,NSM,EPR,LRR,WLR
0,Auckland,Hobsonville,1940-04-22,143.751148,2.196222,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,POINT (1749229.335 5926802.329),100227997741,0.000000,-8.68,-0.10,-0.14,-0.12
1,Auckland,Hobsonville,1950-10-14,145.625242,2.148497,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,POINT (1749230.848 5926803.435),100227997741,10.477755,-8.68,-0.10,-0.14,-0.12
2,Auckland,Hobsonville,2017-01-01,132.473747,0.436492,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,POINT (1749220.23 5926795.676),100227997741,76.695414,-8.68,-0.10,-0.14,-0.12
3,Auckland,Hobsonville,2024-01-01,135.066520,0.436492,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,POINT (1749222.323 5926797.206),100227997741,83.693361,-8.68,-0.10,-0.14,-0.12
4,Auckland,Hobsonville,1940-04-22,142.676656,2.196222,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,POINT (1749234.367 5926793.621),100227998951,0.000000,-9.52,-0.11,-0.14,-0.12
5,Auckland,Hobsonville,1950-10-14,143.259288,2.148497,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,POINT (1749234.837 5926793.964),100227998951,10.477755,-9.52,-0.11,-0.14,-0.12
6,Auckland,Hobsonville,2017-01-01,131.468844,0.436492,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,POINT (1749225.317 5926787.008),100227998951,76.695414,-9.52,-0.11,-0.14,-0.12
7,Auckland,Hobsonville,2024-01-01,133.153426,0.436492,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,POINT (1749226.677 5926788.002),100227998951,83.693361,-9.52,-0.11,-0.14,-0.12
8,Auckland,Hobsonville,1940-04-22,140.677433,2.196222,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,POINT (1749238.651 5926784.366),100227999629,0.000000,-9.36,-0.11,-0.13,-0.12
9,Auckland,Hobsonville,1950-10-14,140.833145,2.148497,Z:\Retrolens\Auckland\Hobsonville\Shorelines\Hobsonville_shorelines.shp,POINT (1749238.777 5926784.458),100227999629,10.477755,-9.36,-0.11,-0.13,-0.12


Rates rows: 2,074
Point rows: 10,995
DSAS exclusions rows: 27
time: 59.5 s (started: 2026-08-19 17:15:01 +12:00)


In [13]:
# 5) Save DSAS outputs
rates.to_file(RATES_OUT)
points.to_file(POINTS_OUT)

# Save exclusions report (shorelines not included in DSAS and why).
if 'dsas_exclusions' in globals():
    dsas_exclusions.to_csv(EXCLUSIONS_OUT, index=False)
else:
    pd.DataFrame(columns=['filename', 'region', 'aoi', 'is_new_shoreline', 'total_rows', 'excluded_rows', 'included_rows', 'rows_without_uncy', 'reason']).to_csv(EXCLUSIONS_OUT, index=False)

# Optional tabular exports for easy QA
rates_csv = RATES_OUT.with_suffix('.csv')
points_csv = POINTS_OUT.with_suffix('.csv')
rates.drop(columns='geometry').to_csv(rates_csv, index=False)
points.drop(columns='geometry').to_csv(points_csv, index=False)

print(f"Saved shorelines shapefile: {V2_PATH}")
print(f"Saved rates shapefile: {RATES_OUT}")
print(f"Saved points shapefile: {POINTS_OUT}")
print(f"Saved exclusions report: {EXCLUSIONS_OUT}")
print(f"Saved CSV companions: {rates_csv.name}, {points_csv.name}")


Saved shorelines shapefile: DataUpdatev2\catriona\NZCCDv2_TeAtatuHobsonvillePollenIslandShoalBaySoldiersBayNgataringaFrenchmansBayManukapuaIslandOmokoritoBayOrongoPointPoutoShellyBeachTeHakonoclarksBayTinopai.shp
Saved rates shapefile: DataUpdatev2\catriona\ratesv2_TeAtatuHobsonvillePollenIslandShoalBaySoldiersBayNgataringaFrenchmansBayManukapuaIslandOmokoritoBayOrongoPointPoutoShellyBeachTeHakonoclarksBayTinopai.shp
Saved points shapefile: DataUpdatev2\catriona\intersectsv2_TeAtatuHobsonvillePollenIslandShoalBaySoldiersBayNgataringaFrenchmansBayManukapuaIslandOmokoritoBayOrongoPointPoutoShellyBeachTeHakonoclarksBayTinopai.shp
Saved exclusions report: DataUpdatev2\catriona\new_dsas_exclusions_TeAtatuHobsonvillePollenIslandShoalBaySoldiersBayNgataringaFrenchmansBayManukapuaIslandOmokoritoBayOrongoPointPoutoShellyBeachTeHakonoclarksBayTinopai.csv
Saved CSV companions: ratesv2_TeAtatuHobsonvillePollenIslandShoalBaySoldiersBayNgataringaFrenchmansBayManukapuaIslandOmokoritoBayOrongoPointPou

c:\envs\coastal\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field Date created as String field, though DateTime requested.
  ogr_write(
c:\envs\coastal\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'YearsSinceBase' to 'YearsSince'
  ogr_write(
